## Libraries

In [0]:
from pyspark.sql.functions import col, avg, when
from pyspark.sql import DataFrame
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import concat_ws
import pandas as pd

## Load data

In [0]:
indexed_data = spark.table('workspace.telco.indexed_data')

In [0]:
display(indexed_data.limit(5))

## Features

I will create features from indexed_data grouping by different categorical columns. 

### Numerical features

In [0]:
categorical_cols = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

In [0]:
indexed_data_df = indexed_data.toPandas()

In [0]:
indexed_data_df["tenure_safe"] = indexed_data_df["tenure"].apply(lambda x: x if x > 0 else None)

def feat_related_charges(df: pd.DataFrame, column:str):
    
    df["monthly_charge_per_tenure"] = df["MonthlyCharges"] / df["tenure_safe"]
    df["total_charge_per_tenure"] = df["TotalCharges"] / df["tenure_safe"]
    
    df[column + "_avg_tenure"] = df.groupby(column)["tenure"].transform("mean")
    df[column + "_avg_monthly_charge"] = df.groupby(column)["MonthlyCharges"].transform("mean")
    df[column + "_avg_total_charge"] = df.groupby(column)["TotalCharges"].transform("mean")

    df[column + "_avg_monthly_charge_per_tenure"] = df.groupby(column)["monthly_charge_per_tenure"].transform("mean")
    df[column + "_avg_total_charge_per_tenure"] = df.groupby(column)["total_charge_per_tenure"].transform("mean")

    return df

for column in categorical_cols:
    indexed_data_df = feat_related_charges(indexed_data_df, column)

In [0]:
indexed_data_df.head()

In [0]:
indexed_data_df.dtypes

## Combining categorical features

In [0]:
for col in categorical_cols:
    indexed_data_df[col] = indexed_data_df[col].astype("object")
indexed_data_df.info()

In [0]:
for col1 in categorical_cols:
    for col2 in categorical_cols:
        if col1 != col2:
            indexed_data_df[f"{col1}_{col2}"] = (
                indexed_data_df[col1].astype(str) + "_" + indexed_data_df[col2].astype(str)
            )
indexed_data_df.head()

## Encoding new categorical fields

In [0]:
new_categorical_names = []
for col1 in categorical_cols:
    for col2 in categorical_cols:
        if col1 != col2:
            new_categorical_names.append(f"{col1}_{col2}")
print(new_categorical_names)

In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in new_categorical_names:
    indexed_data_df[col] = le.fit_transform(indexed_data_df[col])

indexed_data_df.head()

In [0]:
indexed_data_df.drop(["tenure_safe"], inplace=True, axis=1)

## Scaling numerical features

In [0]:
numerical_cols = indexed_data_df.select_dtypes(include=[int, float]).columns
final_num_cols = list(set(numerical_cols) - set(categorical_cols) - set(new_categorical_names) - set(["Churn"]))

In [0]:
final_num_cols

In [0]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = pd.DataFrame(scaler.fit_transform(indexed_data_df[final_num_cols]))
scaled_data.columns = final_num_cols
scaled_data.head()

In [0]:
indexed_tmp = indexed_data_df.drop(final_num_cols, axis=1)
indexed_tmp.head()

In [0]:
scaled_data = pd.concat([indexed_tmp, scaled_data], axis=1)
scaled_data.head()

In [0]:
# Saving scaling and encoding data
scaled_data = spark.createDataFrame(scaled_data)

In [0]:
scaled_data.write.mode("overwrite").saveAsTable("workspace.telco.ml_silver_data")